In [7]:
import glob, os, subprocess, time, zipfile
HOME = "/home/sagemaker-user"
busy = subprocess.run(["pgrep", "-af", "run_pipeline.py|-m mctsjssp"], capture_output=True, text=True).stdout.strip()
assert not busy, "something is still running - stop it first:\n" + busy
zips = sorted(glob.glob(f"{HOME}/**/mcts_jssp_untrained_24nov_v2*.zip", recursive=True), key=os.path.getmtime)
assert zips, "upload mcts_jssp_untrained_24nov_v2.zip first"
z = zipfile.ZipFile(zips[-1])                      # opened before the rename, so it works wherever you uploaded it
os.chdir(HOME)
if os.path.exists("mcts_jssp"):
    old = f"mcts_jssp_old_{time.strftime('%Y%m%d_%H%M')}"
    os.rename("mcts_jssp", old)
    print("old folder kept as", old)
z.extractall(HOME)
print("unzipped", zips[-1], "->", f"{HOME}/mcts_jssp")

unzipped /home/sagemaker-user/mcts_jssp_untrained_24nov_v2.zip -> /home/sagemaker-user/mcts_jssp


In [9]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"
os.chdir("/home/sagemaker-user/mcts_jssp")
exec(open("sagemaker_pilot.py").read())


>>> /opt/conda/bin/python -m pip install -q torch --index-url https://download.pytorch.org/whl/cpu

>>> /opt/conda/bin/python -m pip install -q torch_geometric

>>> /opt/conda/bin/python -m pip install -q -e .[test]
48 vCPUs, 24 physical cores -> 24 workers

>>> /opt/conda/bin/python -m pytest -q -m not slow
........................................................................ [ 32%]
........................................................................ [ 65%]
........................................................................ [ 97%]
.....                                                                    [100%]
221 passed, 12 deselected in 33.69s

>>> /opt/conda/bin/python run_pipeline.py --steps validate,pilot
settings: study.yaml   workers: 24   steps: validate, pilot

=== validate ====================================================

>>> python -m mctsjssp validate-env --config configs/base.yaml --set seed=3 --set runs=5 --set search.c=1.0 --set search.skip_forced=true -

In [16]:
running()
show("pilot", 12)

nothing running
az_15x15_s020    ffnn 15x15    0.2             0.04411                     0.05612                 0.2245
      flr_6x6 fast_lr   6x6    NaN           0.0005848                    0.002674                0.06686
    flr_10x10 fast_lr 10x10    NaN            0.004141                    0.008789                 0.0791
    flr_15x15 fast_lr 15x15    NaN             0.01872                     0.02439                0.09756
      faz_6x6 fast_az   6x6    NaN           0.0005848                    0.002482                0.06204
    faz_10x10 fast_az 10x10    NaN            0.004141                     0.00854                0.07686
    faz_15x15 fast_az 15x15    NaN             0.01872                     0.02331                0.09323

Tables written to /home/sagemaker-user/mcts_jssp/results/pilot
    (3.1 min)

Done in 0.05 h.  Results: results/, analyses: results/<experiment>/analysis/, comparisons: results/compare/ and results/visited_states/



In [17]:
import os, subprocess, sys, shutil, importlib.util
os.chdir("/home/sagemaker-user/mcts_jssp")
print("folder:", os.getcwd())
os.makedirs("logs", exist_ok=True)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# a restarted space can lose installed packages: install them again if missing
missing = [m for m in ("torch", "torch_geometric") if importlib.util.find_spec(m) is None]
if missing and os.path.exists("logs/pilot.log"):
    print("installing", ", ".join(missing), "...")
    pip = [sys.executable, "-m", "pip", "install", "-q"]
    subprocess.run(pip + ["torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    subprocess.run(pip + ["torch_geometric"], check=True)
    subprocess.run(pip + ["-e", ".[test]"], check=True)

def start(steps, log, hours=None):
    if running(quiet=True):
        print("a stage is still running - wait until it has finished")
        return
    cmd = [sys.executable, "run_pipeline.py", "--steps", steps] + (["--hours", str(hours)] if hours else [])
    p = subprocess.Popen(cmd, stdout=open(f"logs/{log}.log", "a"), stderr=subprocess.STDOUT,
                         env=dict(os.environ), start_new_session=True)
    print(f"started: {steps}  (process {p.pid}, log logs/{log}.log)")

def show(log, lines=25):
    path = f"logs/{log}.log"
    print("".join(open(path).readlines()[-lines:]) if os.path.exists(path) else f"no {path} yet")

def running(quiet=False):
    out = subprocess.run(["pgrep", "-af", "run_pipeline.py|sagemaker_pilot.py"], capture_output=True, text=True).stdout.strip()
    if not quiet:
        print(out or "nothing running")
    return bool(out)

print("ready")

folder: /home/sagemaker-user/mcts_jssp
ready


In [18]:
start("train,check,self_play,compare_training,select_entropy,disturbed", "prepare")

started: train,check,self_play,compare_training,select_entropy,disturbed  (process 1963, log logs/prepare.log)


In [19]:
running()
show("prepare")

nothing running
-> /home/sagemaker-user/mcts_jssp/results/training/disturbed_distribution.png
      network variant  sigma  iteration  best_iteration     stopped_by
az_10x10_s020      FT    0.2         19              -1 no_improvement
  az_6x6_s010      FT    0.1         19               7 no_improvement
az_10x10_s010      FT    0.1         19              -1 no_improvement
lr_15x15_s020      FT    0.2         19               1 no_improvement
az_15x15_s020      FT    0.2         19              -1 no_improvement
az_15x15_s010      FT    0.1         19              -1 no_improvement
lr_10x10_s020      SC    0.2         19               1 no_improvement
az_10x10_s020      SC    0.2         21              11 no_improvement
  az_6x6_s020      FT    0.2         22              12 no_improvement
lr_15x15_s020      SC    0.2         22              12 no_improvement
  lr_6x6_s010      SC    0.1         25              15 no_improvement
lr_15x15_s010      FT    0.1         27              1

In [20]:
import glob, os, zipfile
os.chdir("/home/sagemaker-user/mcts_jssp")
files = (glob.glob("results/pilot/*") + glob.glob("results/validation/*") + glob.glob("results/training/**/*", recursive=True)
         + glob.glob("logs/*.log") + glob.glob("artifacts/*.json") + glob.glob("artifacts/models/*/*.json")
         + glob.glob("artifacts/scratch/models/*/*.json") + glob.glob("artifacts/self_play/models/*/*.json"))
files = sorted(f for f in set(files) if os.path.isfile(f))
with zipfile.ZipFile("preparation_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f)
print(f"{len(files)} files in {os.path.abspath('preparation_results.zip')} ({os.path.getsize('preparation_results.zip')/1e6:.1f} MB)")

122 files in /home/sagemaker-user/mcts_jssp/preparation_results.zip (4.3 MB)


In [21]:
start("run", "experiments")

started: run  (process 8141, log logs/experiments.log)


In [23]:
running()
show("experiments")

8141 /opt/conda/bin/python run_pipeline.py --steps run
  328/3280 runs, 0.1 min elapsed, about 1.2 min left
  492/3280 runs, 0.2 min elapsed, about 1.1 min left
  656/3280 runs, 0.3 min elapsed, about 1.3 min left
  820/3280 runs, 0.6 min elapsed, about 1.9 min left
  984/3280 runs, 0.8 min elapsed, about 1.9 min left
  1148/3280 runs, 0.9 min elapsed, about 1.6 min left
  1312/3280 runs, 1.1 min elapsed, about 1.6 min left
  1476/3280 runs, 2.1 min elapsed, about 2.6 min left
  1640/3280 runs, 3.0 min elapsed, about 3.0 min left
  1804/3280 runs, 3.1 min elapsed, about 2.6 min left
  1968/3280 runs, 3.3 min elapsed, about 2.2 min left
  2132/3280 runs, 3.5 min elapsed, about 1.9 min left
  2296/3280 runs, 4.2 min elapsed, about 1.8 min left
  2460/3280 runs, 6.0 min elapsed, about 2.0 min left
  2624/3280 runs, 6.8 min elapsed, about 1.7 min left
  2788/3280 runs, 7.1 min elapsed, about 1.2 min left
  2952/3280 runs, 8.2 min elapsed, about 0.9 min left
  3116/3280 runs, 15.6 min elaps

In [72]:
import glob, json, os, subprocess, time
os.chdir("/home/sagemaker-user/mcts_jssp")
procs = subprocess.run(["pgrep", "-af", "run_pipeline.py|-m mctsjssp run"], capture_output=True, text=True).stdout.strip()
print("RUNNING:\n" + procs if procs else "NOTHING RUNNING")
print()
for meta in sorted(glob.glob("results/*/meta.json"), key=os.path.getmtime):
    d = os.path.dirname(meta)
    planned = json.load(open(meta))["runs_planned"]
    f = os.path.join(d, "results.jsonl")
    done = sum(1 for _ in open(f)) if os.path.exists(f) else 0
    last = time.strftime("%H:%M", time.localtime(os.path.getmtime(f))) if done else "-"
    print(f"{os.path.basename(d):28s} {done:5d}/{planned} runs ({done/planned:.0%}), last result {last}")
print("\nnow:", time.strftime("%H:%M"))

NOTHING RUNNING

rq1-0dbc859e                  3280/3280 runs (100%), last result 13:03
rq2-c5211bf5                  7380/7380 runs (100%), last result 21:17
rq3-627f09ef                  3120/3120 runs (100%), last result 22:21
rq4-972d6c55                  2050/2050 runs (100%), last result 22:30
rq5-d8cd2ac6                  1080/1080 runs (100%), last result 22:41
fast-b5c6bddb                 2640/2640 runs (100%), last result 23:16
sensitivity_c-5fead424         720/720 runs (100%), last result 23:56
rq6-b71cd023                   600/600 runs (100%), last result 00:21

now: 00:22


In [ ]:
import os, shutil
os.chdir("/home/sagemaker-user/mcts_jssp")
for name, folder in [("study_results", "results"), ("study_logs", "logs")]:
    shutil.make_archive(name, "zip", ".", folder)
    print(f"{name}.zip  ({os.path.getsize(name + '.zip') / 1e6:.1f} MB)")

In [73]:
import os, shutil, subprocess, time, zipfile
HOME = "/home/sagemaker-user"
SRC = f"{HOME}/mcts_jssp"
busy = subprocess.run(["pgrep", "-af", "run_pipeline.py|-m mctsjssp"], capture_output=True, text=True).stdout.strip()
assert not busy, "something is still running - wait until it has finished:\n" + busy
size = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(SRC) for f in fs) / 1e9
print(f"folder: {size:.1f} GB, free disk: {shutil.disk_usage(HOME).free / 1e9:.1f} GB")
out = f"{HOME}/mcts_jssp_complete_{time.strftime('%Y%m%d_%H%M')}.zip"
skip = ("__pycache__", ".ipynb_checkpoints", ".pytest_cache")
n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for root, dirs, files in os.walk(SRC):
        dirs[:] = [d for d in dirs if d not in skip]
        for f in files:
            path = os.path.join(root, f)
            if f.endswith(".zip"):          # earlier result zips inside the folder: not needed twice
                continue
            z.write(path, os.path.relpath(path, HOME))
            n += 1
print(f"{n} files -> {out} ({os.path.getsize(out) / 1e9:.2f} GB)")

folder: 0.9 GB, free disk: 4.1 GB
41038 files -> /home/sagemaker-user/mcts_jssp_complete_20261125_0034.zip (0.79 GB)


In [4]:
import glob, hashlib, os, py_compile, shutil, subprocess, sys, time, zipfile
HOME = "/home/sagemaker-user"
DST = f"{HOME}/mcts_jssp"
FILES = ["README.md", "configs/experiments/rules.yaml", "run_pipeline.py",
         "src/mctsjssp/analysis/plots.py", "src/mctsjssp/analysis/report.py", "src/mctsjssp/analysis/rq.py",
         "src/mctsjssp/cli.py", "src/mctsjssp/distributions/__init__.py", "src/mctsjssp/distributions/fixed.py",
         "src/mctsjssp/experiments/artifacts.py", "src/mctsjssp/experiments/design.py",
         "src/mctsjssp/experiments/pilot.py", "src/mctsjssp/experiments/runner.py",
         "src/mctsjssp/experiments/time_budget.py", "src/mctsjssp/training/tuning.py",
         "tests/test_experiments.py", "tests/test_rules.py"]

# 0. nothing may be running
busy = subprocess.run(["pgrep", "-af", "run_pipeline.py|-m mctsjssp"], capture_output=True, text=True).stdout.strip()
assert not busy, "something is still running - do NOT stop it; run this cell when it has finished:\n" + busy

# 1. the new zip, whatever its name: it holds the rules experiment
def is_new(path):
    try:
        with zipfile.ZipFile(path) as z:
            return "mcts_jssp/configs/experiments/rules.yaml" in z.namelist()
    except (OSError, zipfile.BadZipFile):
        return False
zips = sorted((p for p in glob.glob(f"{HOME}/**/*.zip", recursive=True) if is_new(p)), key=os.path.getmtime)
assert zips, "the new zip is not on SageMaker yet - upload mcts_jssp_rules_patch_25nov.zip and run this cell again"
ZIP = zips[-1]

# 2. backup: the whole folder if it fits comfortably on the disk, otherwise everything but results/
#    (the patch does not touch results/)
def size_of(path):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs)
total, results = size_of(DST), size_of(f"{DST}/results") if os.path.isdir(f"{DST}/results") else 0
free = shutil.disk_usage(HOME).free
backup = f"{HOME}/mcts_jssp_backup_{time.strftime('%Y%m%d_%H%M')}"
if free > 1.2 * total + 2e9:
    shutil.copytree(DST, backup, symlinks=True)
    print(f"backup: {backup} (whole folder, {total / 1e9:.2f} GB)")
else:
    shutil.copytree(DST, backup, symlinks=True, ignore=lambda d, names: ["results"] if d == DST else [])
    print(f"backup: {backup} (everything but results/, {(total - results) / 1e9:.2f} GB; "
          f"not enough free disk for the {results / 1e9:.1f} GB of results, which the patch does not touch)")

# 3. the changed and new files (study.yaml, your notebooks, networks and results stay as they are)
with zipfile.ZipFile(ZIP) as z:
    for f in FILES:
        data = z.read(f"mcts_jssp/{f}")
        os.makedirs(os.path.dirname(f"{DST}/{f}") or DST, exist_ok=True)
        with open(f"{DST}/{f}.new", "wb") as fh:
            fh.write(data)
        os.replace(f"{DST}/{f}.new", f"{DST}/{f}")
        assert hashlib.sha1(open(f"{DST}/{f}", "rb").read()).digest() == hashlib.sha1(data).digest()
        if f.endswith(".py"):
            py_compile.compile(f"{DST}/{f}", doraise=True)
print(f"updated {len(FILES)} files from {ZIP}")

# 4. the finished experiments are found where they are (nothing will run again)
check = r'''
import importlib.util, json, sys, yaml
sys.path.insert(0, "src")
spec = importlib.util.spec_from_file_location("rp", "run_pipeline.py"); rp = importlib.util.module_from_spec(spec); spec.loader.exec_module(rp)
from mctsjssp.experiments.config import load_config
from mctsjssp.experiments.design import plan
study = yaml.safe_load(open("study.yaml"))
runner = type("R", (), {"study": study})()
for e in study["pipeline"]["experiments"]:
    d = rp.Runner.run_dir(runner, e)
    cfg = load_config(rp.EXPERIMENTS / f"{e}.yaml", [a for a in rp.set_args(rp.overrides_for(study, e)) if a != "--set"])
    planned = len(plan(cfg))
    f = d / "results.jsonl"
    done = sum(1 for _ in open(f)) if f.exists() else 0
    print(f"  {e:14s} {d.name:24s} {done:5d}/{planned} runs" + ("" if done >= planned else "   <- NOT COMPLETE"))
'''
res = subprocess.run([sys.executable, "-c", check], cwd=DST, env={**os.environ, "PYTHONPATH": "src"},
                     capture_output=True, text=True)
print("finished experiments:\n" + (res.stdout or res.stderr[-1500:]))

# 5. quick check of the new code (under a minute)
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_rules.py"],
                     cwd=DST, env={**os.environ, "PYTHONPATH": "src"}, capture_output=True, text=True)
print(res.stdout.strip().splitlines()[-1] if res.stdout.strip() else res.stderr[-2000:])
print("PATCH OK" if res.returncode == 0 else "TESTS FAILED - send me the output above")

backup: /home/sagemaker-user/mcts_jssp_backup_20261125_0743 (whole folder, 0.86 GB)
updated 17 files from /home/sagemaker-user/mcts_jssp_rules_patch_25nov.zip
finished experiments:
Traceback (most recent call last):
  File "<string>", line 5, in <module>
    from mctsjssp.experiments.config import load_config
  File "/home/sagemaker-user/mcts_jssp/src/mctsjssp/experiments/__init__.py", line 2, in <module>
    from .artifacts import Artifacts
  File "/home/sagemaker-user/mcts_jssp/src/mctsjssp/experiments/artifacts.py", line 25, in <module>
    from ..networks.checkpoint import file_fingerprint, load_checkpoint
  File "/home/sagemaker-user/mcts_jssp/src/mctsjssp/networks/__init__.py", line 2, in <module>
    from .checkpoint import build_model, file_fingerprint, load_checkpoint, save_checkpoint
  File "/home/sagemaker-user/mcts_jssp/src/mctsjssp/networks/checkpoint.py", line 12, in <module>
    from .gin import GINPolicyValue
  File "/home/sagemaker-user/mcts_jssp/src/mctsjssp/networks/

In [5]:
import importlib.util, os, subprocess, sys
os.chdir("/home/sagemaker-user/mcts_jssp")

# 1. packages removed by the restart of the space: the same commands as stage 1
missing = [m for m in ("torch", "torch_geometric") if importlib.util.find_spec(m) is None]
print("missing:", ", ".join(missing) or "nothing")
pip = [sys.executable, "-m", "pip", "install", "-q"]
if "torch" in missing:
    subprocess.run(pip + ["torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
if "torch_geometric" in missing:
    subprocess.run(pip + ["torch_geometric"], check=True)
subprocess.run(pip + ["-e", ".[test]"], check=True)
print(subprocess.run([sys.executable, "-c", "import torch, torch_geometric; print('torch', torch.__version__, "
                      "'| torch_geometric', torch_geometric.__version__)"], capture_output=True, text=True).stdout)

# 2. the finished experiments are found where they are (nothing will run again)
check = r'''
import importlib.util, sys, yaml
sys.path.insert(0, "src")
spec = importlib.util.spec_from_file_location("rp", "run_pipeline.py"); rp = importlib.util.module_from_spec(spec); spec.loader.exec_module(rp)
from mctsjssp.experiments.config import load_config
from mctsjssp.experiments.design import plan
study = yaml.safe_load(open("study.yaml"))
runner = type("R", (), {"study": study})()
for e in study["pipeline"]["experiments"]:
    d = rp.Runner.run_dir(runner, e)
    cfg = load_config(rp.EXPERIMENTS / f"{e}.yaml", [a for a in rp.set_args(rp.overrides_for(study, e)) if a != "--set"])
    planned = len(plan(cfg))
    f = d / "results.jsonl"
    done = sum(1 for _ in open(f)) if f.exists() else 0
    print(f"  {e:14s} {d.name:24s} {done:5d}/{planned} runs" + ("" if done >= planned else "   <- NOT COMPLETE"))
'''
res = subprocess.run([sys.executable, "-c", check], env={**os.environ, "PYTHONPATH": "src"}, capture_output=True, text=True)
print("finished experiments:\n" + (res.stdout or res.stderr[-1500:]))

# 3. quick check of the new code
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_rules.py"],
                     env={**os.environ, "PYTHONPATH": "src"}, capture_output=True, text=True)
print(res.stdout.strip().splitlines()[-1] if res.stdout.strip() else res.stderr[-2000:])
print("PATCH OK" if res.returncode == 0 else "TESTS FAILED - send me the output above")

missing: torch_geometric
torch 2.12.0 | torch_geometric 2.8.1

finished experiments:
  rq1            rq1-0dbc859e              3280/3280 runs
  rq2            rq2-c5211bf5              7380/7380 runs
  rq3            rq3-627f09ef              3120/3120 runs
  rq4            rq4-972d6c55              2050/2050 runs
  rq5            rq5-d8cd2ac6              1080/1080 runs
  fast           fast-b5c6bddb             2640/2640 runs
  sensitivity_c  sensitivity_c-5fead424     720/720 runs
  rq6            rq6-b71cd023               600/600 runs

10 passed in 1.31s
PATCH OK


In [8]:
import os, subprocess, sys, importlib.util
os.chdir("/home/sagemaker-user/mcts_jssp")
print("folder:", os.getcwd())
os.makedirs("logs", exist_ok=True)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"          # progress lines reach the log at once

# a restarted space loses installed packages: install them again if missing
missing = [m for m in ("torch", "torch_geometric") if importlib.util.find_spec(m) is None]
if missing:
    print("installing", ", ".join(missing), "...")
    pip = [sys.executable, "-m", "pip", "install", "-q"]
    if "torch" in missing:
        subprocess.run(pip + ["torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    subprocess.run(pip + ["torch_geometric"], check=True)
    subprocess.run(pip + ["-e", ".[test]"], check=True)

def start(steps, log, hours=None):
    if running(quiet=True):
        print("a stage is still running - wait until it has finished")
        return
    cmd = [sys.executable, "run_pipeline.py", "--steps", steps] + (["--hours", str(hours)] if hours else [])
    p = subprocess.Popen(cmd, stdout=open(f"logs/{log}.log", "a"), stderr=subprocess.STDOUT,
                         env=dict(os.environ), start_new_session=True)
    print(f"started: {steps}  (process {p.pid}, log logs/{log}.log)")

def show(log, lines=25):
    path = f"logs/{log}.log"
    print("".join(open(path).readlines()[-lines:]) if os.path.exists(path) else f"no {path} yet")

def running(quiet=False):
    out = subprocess.run(["pgrep", "-af", "run_pipeline.py|sagemaker_pilot.py"], capture_output=True, text=True).stdout.strip()
    if not quiet:
        print(out or "nothing running")
    return bool(out)

print("ready")

folder: /home/sagemaker-user/mcts_jssp
ready


In [9]:
start("rules,fixed_states,analyse,compare", "analysis")

started: rules,fixed_states,analyse,compare  (process 486, log logs/analysis.log)


In [18]:
running()
show("analysis")

nothing running

>>> python -m mctsjssp compare-states --run-dir /home/sagemaker-user/mcts_jssp/results/rq1-0dbc859e --run-dir /home/sagemaker-user/mcts_jssp/results/rq2-c5211bf5 --run-dir /home/sagemaker-user/mcts_jssp/results/rq3-627f09ef --run-dir /home/sagemaker-user/mcts_jssp/results/rq4-972d6c55 --run-dir /home/sagemaker-user/mcts_jssp/results/fast-b5c6bddb --out results/visited_states
  rq1-0dbc859e: 0 runs with stored rollout sequences
  rq2-c5211bf5: 7371 runs with stored rollout sequences
  rq3-627f09ef: 3120 runs with stored rollout sequences
  rq4-972d6c55: 0 runs with stored rollout sequences
  fast-b5c6bddb: 2640 runs with stored rollout sequences
visited_states: 26004 rows -> results/visited_states/visited_states.csv
visited_summary: 660 rows -> results/visited_states/visited_summary.csv
visited_overlap: 178192 rows -> results/visited_states/visited_overlap.csv
figure -> results/visited_states/visited_states_I25.png
figure -> results/visited_states/visited_states_I25.pdf

In [19]:
import os, time, zipfile
os.chdir("/home/sagemaker-user/mcts_jssp")
out = f"/home/sagemaker-user/mcts_jssp_all_results_{time.strftime('%Y%m%d_%H%M')}.zip"
skip = ("__pycache__", ".ipynb_checkpoints", ".pytest_cache")
n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for top in ("results", "logs", "artifacts", "configs"):
        for root, dirs, files in os.walk(top):
            dirs[:] = [d for d in dirs if d not in skip]
            for f in files:
                if not f.endswith(".zip"):
                    z.write(os.path.join(root, f))
                    n += 1
    z.write("study.yaml"); n += 1
print(f"{n} files -> {out} ({os.path.getsize(out) / 1e6:.0f} MB)")

48428 files -> /home/sagemaker-user/mcts_jssp_all_results_20261125_0915.zip (999 MB)
